# Multi-Task DNN
Train the multi-task DNN with three heads.

In [ ]:
from models.dnn_model import build_multitask_dnn, train_dnn
from training.evaluate import stratified_split
from preprocessing.scaler import fit_and_scale
import pandas as pd
from config import PROCESSED_DATA_PATH, FEATURE_COLS

df = pd.read_csv(f'{PROCESSED_DATA_PATH}merged_soil_data.csv')
available = [c for c in FEATURE_COLS if c in df.columns]
X = df[available].values
y_crop = df['crop'].values
y_fert = df['fertility_grade'].values
y_def = df['nutrient_status'].values

X_train, X_val, X_test, yc_train, yc_val, yc_test = stratified_split(X, y_crop)
_, _, _, yf_train, yf_val, _ = stratified_split(X, y_fert)
_, _, _, yd_train, yd_val, _ = stratified_split(X, y_def)

X_train_s, X_val_s, X_test_s, _ = fit_and_scale(X_train, X_val, X_test)
model = build_multitask_dnn(X_train_s.shape[1], num_crops=len(set(y_crop)))
train_dnn(model, X_train_s, yc_train, yf_train, yd_train, X_val_s, yc_val, yf_val, yd_val)